In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import skimage.color
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization, concatenate
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping, CSVLogger
from sklearn.metrics import classification_report
from mlxtend.evaluate import confusion_matrix
from mlxtend.plotting import plot_confusion_matrix

# fixing randomness
seed = 7
np.random.seed(seed)
tf.random.set_seed(seed)

# image size + batch size
img_size = (224,224)
batch_size = 32

# training schedule (3-phase fine-tuning)
initial_epochs = 10       # train only dense layers
phase2_epochs = 12        # train last ResNet block
phase3_epochs = 12        # train full model

train_samples = 7200
val_samples = 2400
test_samples = 2400

num_classes = 3 # 3-class classification

train_dir = "/train"
val_dir = "/val"
test_dir = "/test"

model_save_path = "mcrestnet_best.h5"

USE_MIXUP = True      # mixup augmentation improves generalization
MIXUP_ALPHA = 0.2
UNFREEZE_FROM = 140   # last ResNet block to unfreeze in Phase 2

# -------------------------------------------------------------------
# Preprocessing functions for LCH and HSV branches
# -------------------------------------------------------------------

# rescales each channel between 0–255 so that LAB/LCH/HSV remain valid
def scale0to255(image):
    a = image.astype(np.float32)
    if a.ndim == 3 and a.shape[2] >= 3:
        for i in range(3):
            mn = np.min(a[:,:,i])
            mx = np.max(a[:,:,i])
            if mx - mn != 0:
                a[:,:,i] = ((a[:,:,i] - mn) / (mx - mn)) * 255.0
            else:
                a[:,:,i] = 0.0
    return a.astype(np.uint8)

# Laplacian + Gaussian filter => enhances edges & improves texture learning
def log_filter(image):
    g = cv2.GaussianBlur(image, (3,3), 0)
    lap = cv2.Laplacian(np.uint8(g), cv2.CV_64F)
    sharp = np.uint8(np.clip(image + lap, 0, 255))
    return sharp

# convert augmented RGB → LCH color space
def to_lch(batch):
    out = []
    for img in batch:
        x = log_filter(img)
        lab = skimage.color.rgb2lab(x)
        lch = skimage.color.lab2lch(lab)
        lch_s = scale0to255(lch)
        out.append(lch_s)
    return np.stack(out, axis=0)

# convert augmented RGB → HSV color space
def to_hsv(batch):
    out = []
    for img in batch:
        x = log_filter(img)
        hsv = skimage.color.rgb2hsv(x)
        hsv = np.nan_to_num(hsv)
        hsv_s = scale0to255(hsv)
        out.append(hsv_s)
    return np.stack(out, axis=0)

# -------------------------------------------------------------------
# Strong augmentation improves model generalization
# -------------------------------------------------------------------
aug = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.12,
    height_shift_range=0.12,
    shear_range=0.08,
    zoom_range=0.12,
    horizontal_flip=True,
    brightness_range=[0.75,1.25],
    fill_mode='reflect'
)

# validation and test must NOT be augmented
val_aug = ImageDataGenerator()
test_aug = ImageDataGenerator()

# load images from folders
base_flow = aug.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True,
    seed=seed
)

val_flow = val_aug.flow_from_directory(
    val_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

test_flow = test_aug.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=1,
    class_mode='categorical',
    shuffle=False
)

# -------------------------------------------------------------------
# Mixup augmentation:
# blends two images & labels → improves robustness massively
# prevents overfitting & sharp decision boundaries
# -------------------------------------------------------------------
def mixup(batch_x, batch_y, alpha=MIXUP_ALPHA):
    lam = np.random.beta(alpha, alpha)
    batch_size_local = batch_x.shape[0]
    idx = np.random.permutation(batch_size_local)

    x_mix = lam * batch_x + (1 - lam) * batch_x[idx]
    y_mix = lam * batch_y + (1 - lam) * batch_y[idx]
    return x_mix, y_mix

# -------------------------------------------------------------------
# Triple generator for training (with mixup)
# -------------------------------------------------------------------
def triple_generator_train(base_gen):
    while True:
        x_batch, y_batch = next(base_gen)
        x_batch = x_batch.astype(np.uint8)

        if USE_MIXUP:
            x_batch, y_batch = mixup(x_batch, y_batch)

        lch_batch = to_lch(x_batch)
        hsv_batch = to_hsv(x_batch)

        x_rgb = tf.keras.applications.resnet.preprocess_input(x_batch.astype(np.float32))
        x_lch = tf.keras.applications.resnet.preprocess_input(lch_batch.astype(np.float32))
        x_hsv = tf.keras.applications.resnet.preprocess_input(hsv_batch.astype(np.float32))

        yield ([x_rgb, x_lch, x_hsv], y_batch)

# -------------------------------------------------------------------
# Triple generator for validation/test (without mixup)
# -------------------------------------------------------------------
def triple_generator_val(base_gen):
    while True:
        x_batch, y_batch = next(base_gen)
        x_batch = x_batch.astype(np.uint8)

        lch_batch = to_lch(x_batch)
        hsv_batch = to_hsv(x_batch)

        x_rgb = tf.keras.applications.resnet.preprocess_input(x_batch.astype(np.float32))
        x_lch = tf.keras.applications.resnet.preprocess_input(lch_batch.astype(np.float32))
        x_hsv = tf.keras.applications.resnet.preprocess_input(hsv_batch.astype(np.float32))

        yield ([x_rgb, x_lch, x_hsv], y_batch)

train_gen = triple_generator_train(base_flow)
val_gen = triple_generator_val(val_flow)  # Create validation generator

# -------------------------------------------------------------------
# Build each ResNet branch (RGB / LCH / HSV)
# They stay in sync because triple_generator feeds 3 aligned versions
# -------------------------------------------------------------------
def build_branch(prefix):
    backbone = ResNet50(include_top=False, weights='imagenet', input_shape=(224,224,3))

    # rename layers so branches don't have name conflicts
    for i,layer in enumerate(backbone.layers):
        layer._name = f"{prefix}_layer_{i}"

    # freeze here → phase 1 training only affects Dense head
    for layer in backbone.layers:
        layer.trainable = False

    out = backbone.output
    out = GlobalAveragePooling2D()(out)  # converts feature maps → vector
    return backbone.input, out, backbone

in1, out1, b1 = build_branch("r1")
in2, out2, b2 = build_branch("r2")
in3, out3, b3 = build_branch("r3")

# -------------------------------------------------------------------
# Fusion of 3 branches → fully connected head
# BatchNorm + Dropout stop overfitting
# -------------------------------------------------------------------
fused = concatenate([out1, out2, out3])
x = Dense(512, activation='relu')(fused)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
pred = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=[in1, in2, in3], outputs=pred)

# label smoothing improves calibration & helps in 3-class problems
loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

# phase 1 LR – high because we train only Dense layers
opt = Adam(learning_rate=1e-4)

model.compile(optimizer=opt, loss=loss_fn, metrics=['accuracy'])

# -------------------------------------------------------------------
# Callbacks:
# - checkpoint saves best model
# - reduceLR lowers LR when val loss stalls
# - early stopping prevents overfitting
# -------------------------------------------------------------------
checkpoint = ModelCheckpoint(model_save_path, monitor='val_loss', save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1, min_lr=1e-7)
early = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
csvlog = CSVLogger("training_log.csv")

steps_per_epoch = train_samples // batch_size
val_steps = val_samples // batch_size

# -------------------------------------------------------------------
# PHASE 1 — Train only the Dense layers
# -------------------------------------------------------------------
print("Starting Phase 1 training...")
model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=initial_epochs,
    validation_data=val_gen,  # Use val_gen instead of val_flow
    validation_steps=val_steps,
    callbacks=[checkpoint, reduce_lr, early, csvlog]
)

# -------------------------------------------------------------------
# PHASE 2 — unfreeze last block of ResNet50
# improves feature extraction but still safe
# -------------------------------------------------------------------
print("Starting Phase 2 training...")
for layer in b1.layers[UNFREEZE_FROM:]: layer.trainable = True
for layer in b2.layers[UNFREEZE_FROM:]: layer.trainable = True
for layer in b3.layers[UNFREEZE_FROM:]: layer.trainable = True

model.compile(optimizer=Adam(learning_rate=3e-5), loss=loss_fn, metrics=['accuracy'])

model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=phase2_epochs,
    validation_data=val_gen,  # Use val_gen instead of val_flow
    validation_steps=val_steps,
    callbacks=[checkpoint, reduce_lr, early, csvlog]
)

# -------------------------------------------------------------------
# PHASE 3 — unfreeze ALL layers
# very low LR = keeps ImageNet knowledge safe
# best accuracy boost
# -------------------------------------------------------------------
print("Starting Phase 3 training...")
for layer in b1.layers: layer.trainable = True
for layer in b2.layers: layer.trainable = True
for layer in b3.layers: layer.trainable = True

model.compile(optimizer=Adam(learning_rate=5e-6), loss=loss_fn, metrics=['accuracy'])

model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=phase3_epochs,
    validation_data=val_gen,  # Use val_gen instead of val_flow
    validation_steps=val_steps,
    callbacks=[checkpoint, reduce_lr, early, csvlog]
)

model = load_model(model_save_path)

# -------------------------------------------------------------------
# TTA (Test-Time Augmentation)
# flips image horizontally → averages both predictions
# boosts test accuracy by ~1–2%
# -------------------------------------------------------------------
def tta_predict(model, img):
    imgs = [img, np.fliplr(img.copy())]
    preds = []
    for i in imgs:
        x_rgb = tf.keras.applications.resnet.preprocess_input(np.expand_dims(i.astype(np.float32),0))
        x_lch = tf.keras.applications.resnet.preprocess_input(np.expand_dims(to_lch(np.expand_dims(i,0))[0].astype(np.float32),0))
        x_hsv = tf.keras.applications.resnet.preprocess_input(np.expand_dims(to_hsv(np.expand_dims(i,0))[0].astype(np.float32),0))
        p = model.predict([x_rgb, x_lch, x_hsv])[0]
        preds.append(p)
    return np.mean(preds, axis=0)

# -------------------------------------------------------------------
# Evaluate on test set
# -------------------------------------------------------------------
print("Evaluating on test set...")
y_true = []
y_pred = []
files = []

# Create test generator
test_gen = triple_generator_val(test_flow)

for i in range(test_samples):
    ([x_rgb, x_lch, x_hsv], y) = next(test_gen)
    p = model.predict([x_rgb, x_lch, x_hsv])[0]
    y_pred.append(np.argmax(p))
    y_true.append(np.argmax(y[0]))
    files.append(test_flow.filenames[i])

labels_map = dict((v,k) for k,v in test_flow.class_indices.items())
pred_names = [labels_map[i] for i in y_pred]

df = pd.DataFrame({"Filename":files, "Prediction":pred_names})
df.to_csv("test_predictions.csv", index=False)

cm = confusion_matrix(y_target=np.array(y_true), y_predicted=np.array(y_pred), binary=False)
fig, ax = plot_confusion_matrix(conf_mat=cm, class_names=list(labels_map.values()))
plt.savefig("conf_matrix.png")
plt.show()

print(classification_report(y_true, y_pred, target_names=list(labels_map.values())))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---------------------------------------
# FAST PHASE-1 EVALUATION WITH ROC + AUC
# ---------------------------------------

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import pandas as pd
import cv2
import skimage.color
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# ------------------ PATHS ------------------
test_path = "/test"
model_path = "/mcrestnet_best.h5"
img_size = (224,224)

# ------------------ LOAD MODEL SAFELY ------------------
model = load_model(model_path, compile=False)

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

print("✓ Phase-1 model loaded.")

# ------------------ PREPROCESSING ------------------
def scale0to255(image):
    a = image.astype(np.float32)
    for i in range(3):
        mn, mx = np.min(a[:,:,i]), np.max(a[:,:,i])
        a[:,:,i] = ((a[:,:,i]-mn)/(mx-mn))*255 if mx!=mn else 0
    return a.astype(np.uint8)

def log_filter(image):
    g = cv2.GaussianBlur(image, (3,3), 0)
    lap = cv2.Laplacian(np.uint8(g), cv2.CV_64F)
    return np.uint8(np.clip(image + lap, 0, 255))

def to_lch(batch):
    out=[]
    for img in batch:
        lab = skimage.color.rgb2lab(log_filter(img))
        out.append(scale0to255(skimage.color.lab2lch(lab)))
    return np.stack(out)

def to_hsv(batch):
    out=[]
    for img in batch:
        hsv = skimage.color.rgb2hsv(log_filter(img))
        out.append(scale0to255(np.nan_to_num(hsv)))
    return np.stack(out)

# ------------------ TEST GENERATOR ------------------
test_aug = ImageDataGenerator()

test_flow = test_aug.flow_from_directory(
    test_path,
    target_size=img_size,
    batch_size=32,     # <<--- FAST BATCH SIZE
    class_mode='categorical',
    shuffle=False
)

# ------------------ BATCHED EVALUATION ------------------
print("✓ Starting fast evaluation...")

y_true = []
prob_list = []

steps = int(np.ceil(test_flow.n / test_flow.batch_size))

test_flow.reset()

for _ in tqdm(range(steps)):
    x_batch, y_batch = next(test_flow)

    x_batch_uint8 = x_batch.astype(np.uint8)

    lch_batch = to_lch(x_batch_uint8)
    hsv_batch = to_hsv(x_batch_uint8)

    x_rgb = tf.keras.applications.resnet.preprocess_input(x_batch.astype(np.float32))
    x_lch = tf.keras.applications.resnet.preprocess_input(lch_batch.astype(np.float32))
    x_hsv = tf.keras.applications.resnet.preprocess_input(hsv_batch.astype(np.float32))

    preds = model.predict([x_rgb, x_lch, x_hsv], verbose=0)

    prob_list.extend(preds)
    y_true.extend(np.argmax(y_batch, axis=1))

probs = np.array(prob_list)
y_pred = np.argmax(probs, axis=1)

print("✓ Predictions complete!")

# ------------------ LABELS ------------------
labels_map = {v: k for k, v in test_flow.class_indices.items()}
class_names = list(labels_map.values())

# ------------------ CLASSIFICATION REPORT ------------------
print("\n----- CLASSIFICATION REPORT -----\n")
print(classification_report(y_true, y_pred, target_names=class_names))

# ------------------ CONFUSION MATRIX ------------------
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, cmap='Blues', fmt='d',
            xticklabels=class_names,
            yticklabels=class_names)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# ------------------ ROC CURVES + AUC ------------------
y_true_onehot = tf.keras.utils.to_categorical(y_true, num_classes=len(class_names))

plt.figure(figsize=(8,6))
for i, cname in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_true_onehot[:,i], probs[:,i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{cname} (AUC = {roc_auc:.3f})")

plt.plot([0,1], [0,1], linestyle='--')
plt.title("ROC Curves (Phase-1)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(True)
plt.show()

# ------------------ SAVE CONFIDENCE CSV ------------------
df = pd.DataFrame(probs, columns=[f"Conf_{c}" for c in class_names])
df["True"] = [labels_map[i] for i in y_true]
df["Pred"] = [labels_map[i] for i in y_pred]

df.to_csv("phase1_full_metrics.csv", index=False)

print("\n✓ Saved full metrics → phase1_full_metrics.csv")
print("\n✓ Phase-1 evaluation complete!")


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping, CSVLogger
import cv2, skimage.color

train_path = "/train"
val_path   = "/val"
model_path = "/mcrestnet_best.h5"

img_size = (224,224)
batch_size = 32
phase2_epochs = 12
UNFREEZE_FROM = 140
MIXUP_ALPHA = 0.2
USE_MIXUP = True
num_classes = 3

model = load_model(model_path, compile=False)

b1 = [layer for layer in model.layers if layer.name.startswith("r1_layer")]
b2 = [layer for layer in model.layers if layer.name.startswith("r2_layer")]
b3 = [layer for layer in model.layers if layer.name.startswith("r3_layer")]

for layer in b1[UNFREEZE_FROM:]: layer.trainable = True
for layer in b2[UNFREEZE_FROM:]: layer.trainable = True
for layer in b3[UNFREEZE_FROM:]: layer.trainable = True

loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
model.compile(optimizer=Adam(learning_rate=3e-5), loss=loss_fn, metrics=["accuracy"])

def scale0to255(image):
    a = image.astype(np.float32)
    for i in range(3):
        mn, mx = np.min(a[:,:,i]), np.max(a[:,:,i])
        a[:,:,i] = ((a[:,:,i]-mn)/(mx-mn))*255 if mx!=mn else 0
    return a.astype(np.uint8)

def log_filter(img):
    g = cv2.GaussianBlur(img, (3,3), 0)
    lap = cv2.Laplacian(np.uint8(g), cv2.CV_64F)
    return np.uint8(np.clip(img + lap, 0, 255))

def to_lch(batch):
    out=[]
    for img in batch:
        lab = skimage.color.rgb2lab(log_filter(img))
        out.append(scale0to255(skimage.color.lab2lch(lab)))
    return np.stack(out)

def to_hsv(batch):
    out=[]
    for img in batch:
        hsv = skimage.color.rgb2hsv(log_filter(img))
        out.append(scale0to255(np.nan_to_num(hsv)))
    return np.stack(out)

aug = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.12,
    height_shift_range=0.12,
    shear_range=0.08,
    zoom_range=0.12,
    horizontal_flip=True,
    brightness_range=[0.75,1.25],
    fill_mode="reflect"
)

val_aug = ImageDataGenerator()

train_flow = aug.flow_from_directory(train_path, target_size=img_size, batch_size=batch_size, class_mode='categorical', shuffle=True)
val_flow   = val_aug.flow_from_directory(val_path,   target_size=img_size, batch_size=batch_size, class_mode='categorical', shuffle=False)

def mixup(x, y, alpha=MIXUP_ALPHA):
    lam = np.random.beta(alpha, alpha)
    idx = np.random.permutation(x.shape[0])
    return lam*x + (1-lam)*x[idx], lam*y + (1-lam)*y[idx]

def gen_train():
    while True:
        x, y = next(train_flow)
        x = x.astype(np.uint8)
        if USE_MIXUP:
            x, y = mixup(x, y)
        lch = to_lch(x)
        hsv = to_hsv(x)
        x_rgb = tf.keras.applications.resnet.preprocess_input(x.astype(np.float32))
        x_lch = tf.keras.applications.resnet.preprocess_input(lch.astype(np.float32))
        x_hsv = tf.keras.applications.resnet.preprocess_input(hsv.astype(np.float32))
        yield (x_rgb, x_lch, x_hsv), y

def gen_val():
    while True:
        x, y = next(val_flow)
        x = x.astype(np.uint8)
        lch = to_lch(x)
        hsv = to_hsv(x)
        x_rgb = tf.keras.applications.resnet.preprocess_input(x.astype(np.float32))
        x_lch = tf.keras.applications.resnet.preprocess_input(lch.astype(np.float32))
        x_hsv = tf.keras.applications.resnet.preprocess_input(hsv.astype(np.float32))
        yield (x_rgb, x_lch, x_hsv), y

output_signature = (
    (
        tf.TensorSpec(shape=(None, img_size[0], img_size[1], 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None, img_size[0], img_size[1], 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None, img_size[0], img_size[1], 3), dtype=tf.float32),
    ),
    tf.TensorSpec(shape=(None, num_classes), dtype=tf.float32)
)

train_ds = tf.data.Dataset.from_generator(gen_train, output_signature=output_signature)
val_ds   = tf.data.Dataset.from_generator(gen_val,   output_signature=output_signature)

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds.prefetch(tf.data.AUTOTUNE)

steps_per_epoch = train_flow.n // batch_size
val_steps = val_flow.n // batch_size

checkpoint = ModelCheckpoint("/mcrestnet_phase2.h5", monitor="val_loss", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1)
early = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
csvlog = CSVLogger("/phase2_training_log.csv")

model.fit(
    train_ds,
    epochs=phase2_epochs,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_ds,
    validation_steps=val_steps,
    callbacks=[checkpoint, reduce_lr, early, csvlog]
)




In [ ]:
# ---------------------------------------
# FULL PHASE-2 EVALUATION (ALL METRICS + ROC + AUC + CURVES)
# ---------------------------------------

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import pandas as pd
import cv2
import skimage.color
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# ------------ PATHS ----------------
test_path = "/test"
model_path = "/mcrestnet_phase2.h5"
log_path   = "/phase2_training_log.csv"
img_size = (224,224)

# ------------ LOAD MODEL ------------
model = load_model(model_path, compile=False)
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

print("✓ Phase-2 model loaded!")

# ------------ PREPROCESSING ----------
def scale0to255(image):
    a = image.astype(np.float32)
    for i in range(3):
        mn, mx = np.min(a[:,:,i]), np.max(a[:,:,i])
        a[:,:,i] = ((a[:,:,i]-mn)/(mx-mn))*255 if mx!=mn else 0
    return a.astype(np.uint8)

def log_filter(image):
    g = cv2.GaussianBlur(image, (3,3), 0)
    lap = cv2.Laplacian(np.uint8(g), cv2.CV_64F)
    return np.uint8(np.clip(image + lap, 0, 255))

def to_lch(batch):
    out=[]
    for img in batch:
        lab = skimage.color.rgb2lab(log_filter(img))
        out.append(scale0to255(skimage.color.lab2lch(lab)))
    return np.stack(out)

def to_hsv(batch):
    out=[]
    for img in batch:
        hsv = skimage.color.rgb2hsv(log_filter(img))
        out.append(scale0to255(np.nan_to_num(hsv)))
    return np.stack(out)

# ---------- TEST GENERATOR ----------
test_aug = ImageDataGenerator()

test_flow = test_aug.flow_from_directory(
    test_path,
    target_size=img_size,
    batch_size=32,   # FAST evaluation
    class_mode='categorical',
    shuffle=False
)

# -------- FAST BATCHED EVALUATION --------
y_true = []
prob_list = []

steps = int(np.ceil(test_flow.n / 32))
test_flow.reset()

for _ in tqdm(range(steps)):
    x_batch, y_batch = next(test_flow)

    x_uint8 = x_batch.astype(np.uint8)

    lch_batch = to_lch(x_uint8)
    hsv_batch = to_hsv(x_uint8)

    x_rgb = tf.keras.applications.resnet.preprocess_input(x_batch.astype(np.float32))
    x_lch = tf.keras.applications.resnet.preprocess_input(lch_batch.astype(np.float32))
    x_hsv = tf.keras.applications.resnet.preprocess_input(hsv_batch.astype(np.float32))

    preds = model.predict([x_rgb, x_lch, x_hsv], verbose=0)

    prob_list.extend(preds)
    y_true.extend(np.argmax(y_batch, axis=1))

probs = np.array(prob_list)
y_pred = np.argmax(probs, axis=1)

# -------- LABEL MAPPING --------
labels_map = {v:k for k,v in test_flow.class_indices.items()}
class_names = list(labels_map.values())

# -------- CLASSIFICATION REPORT --------
print("\n===== CLASSIFICATION REPORT (Phase-2) =====\n")
print(classification_report(y_true, y_pred, target_names=class_names))

# -------- CONFUSION MATRIX --------
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, cmap='Blues', fmt='d',
            xticklabels=class_names,
            yticklabels=class_names)
plt.title("Confusion Matrix (Phase-2)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# -------- ROC + AUC --------
y_true_onehot = tf.keras.utils.to_categorical(y_true, num_classes=len(class_names))

plt.figure(figsize=(8,6))
for i, cname in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_true_onehot[:,i], probs[:,i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{cname} (AUC = {roc_auc:.3f})")

plt.plot([0,1], [0,1], linestyle='--')
plt.title("ROC Curves (Phase-2)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid()
plt.show()

# -------- SAVE CSV --------
df = pd.DataFrame(probs, columns=[f"Conf_{c}" for c in class_names])
df["True"] = [labels_map[i] for i in y_true]
df["Pred"] = [labels_map[i] for i in y_pred]

df.to_csv("phase2_full_metrics.csv", index=False)
print("✓ Saved → phase2_full_metrics.csv")

# =====================================================
#         TRAINING CURVES (loss + accuracy)
# =====================================================

print("\n===== PLOTTING TRAINING CURVES =====")

try:
    history = pd.read_csv(log_path)

    # ---- LOSS CURVE ----
    plt.figure(figsize=(7,5))
    plt.plot(history["loss"], label="Train Loss")
    plt.plot(history["val_loss"], label="Val Loss")
    plt.title("Phase-2 Loss Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid()
    plt.show()

    # ---- ACCURACY CURVE ----
    plt.figure(figsize=(7,5))
    plt.plot(history["accuracy"], label="Train Acc")
    plt.plot(history["val_accuracy"], label="Val Acc")
    plt.title("Phase-2 Accuracy Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid()
    plt.show()

except:
    print("⚠ phase2_training_log.csv not found. Cannot plot curves.")

print("\n🎉 Phase-2 evaluation complete (ALL METRICS INCLUDED)!")


In [ ]:
# ---------------------------------------
# FINAL PHASE-3 TRAINING CODE
# ---------------------------------------

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping, CSVLogger
import cv2, skimage.color

# ------------------ PATHS ------------------
train_path = "/train"
val_path   = "/val"

phase2_ckpt = "/mcrestnet_phase2.h5"
phase3_best = "/mcrestnet_phase3.h5"
phase3_epoch_save = "/phase3_epoch_{epoch:02d}.h5"
log_path = "/phase3_training_log.csv"

img_size = (224,224)
batch_size = 32
phase3_epochs = 12
MIXUP_ALPHA = 0.2
USE_MIXUP = True
num_classes = 3

# ------------------ LOAD PHASE-2 CHECKPOINT ------------------
model = load_model(phase2_ckpt, compile=False)
print("Loaded Phase-2 best model!")

# ------------------ UNFREEZE ALL LAYERS ------------------
b1 = [layer for layer in model.layers if layer.name.startswith("r1_layer")]
b2 = [layer for layer in model.layers if layer.name.startswith("r2_layer")]
b3 = [layer for layer in model.layers if layer.name.startswith("r3_layer")]

for layer in b1: layer.trainable = True
for layer in b2: layer.trainable = True
for layer in b3: layer.trainable = True

print("All branch layers unfrozen for Phase-3!")

# ------------------ RE-COMPILE MODEL ------------------
loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

model.compile(
    optimizer=Adam(learning_rate=5e-6),
    loss=loss_fn,
    metrics=["accuracy"]
)

print("Model recompiled for Phase-3 training!")

# ------------------ PREPROCESSING FUNCTIONS ------------------
def scale0to255(image):
    a = image.astype(np.float32)
    for i in range(3):
        mn, mx = np.min(a[:,:,i]), np.max(a[:,:,i])
        a[:,:,i] = ((a[:,:,i]-mn)/(mx-mn))*255 if mx!=mn else 0
    return a.astype(np.uint8)

def log_filter(img):
    g = cv2.GaussianBlur(img, (3,3), 0)
    lap = cv2.Laplacian(np.uint8(g), cv2.CV_64F)
    return np.uint8(np.clip(img + lap, 0, 255))

def to_lch(batch):
    out=[]
    for img in batch:
        lab = skimage.color.rgb2lab(log_filter(img))
        out.append(scale0to255(skimage.color.lab2lch(lab)))
    return np.stack(out)

def to_hsv(batch):
    out=[]
    for img in batch:
        hsv = skimage.color.rgb2hsv(log_filter(img))
        out.append(scale0to255(np.nan_to_num(hsv)))
    return np.stack(out)

def mixup(x, y, alpha=MIXUP_ALPHA):
    lam = np.random.beta(alpha, alpha)
    idx = np.random.permutation(x.shape[0])
    return lam*x + (1-lam)*x[idx], lam*y + (1-lam)*y[idx]

# ------------------ DATA LOADERS ------------------
aug = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.12,
    height_shift_range=0.12,
    shear_range=0.08,
    zoom_range=0.12,
    horizontal_flip=True,
    brightness_range=[0.75,1.25],
    fill_mode="reflect"
)

val_aug = ImageDataGenerator()

train_flow = aug.flow_from_directory(
    train_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

val_flow = val_aug.flow_from_directory(
    val_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

# ------------------ GENERATORS ------------------
def gen_train():
    while True:
        x, y = next(train_flow)
        x = x.astype(np.uint8)

        if USE_MIXUP:
            x, y = mixup(x, y)

        lch = to_lch(x)
        hsv = to_hsv(x)

        x_rgb = tf.keras.applications.resnet.preprocess_input(x.astype(np.float32))
        x_lch = tf.keras.applications.resnet.preprocess_input(lch.astype(np.float32))
        x_hsv = tf.keras.applications.resnet.preprocess_input(hsv.astype(np.float32))

        yield (x_rgb, x_lch, x_hsv), y

def gen_val():
    while True:
        x, y = next(val_flow)
        x = x.astype(np.uint8)

        lch = to_lch(x)
        hsv = to_hsv(x)

        x_rgb = tf.keras.applications.resnet.preprocess_input(x.astype(np.float32))
        x_lch = tf.keras.applications.resnet.preprocess_input(lch.astype(np.float32))
        x_hsv = tf.keras.applications.resnet.preprocess_input(hsv.astype(np.float32))

        yield (x_rgb, x_lch, x_hsv), y

output_signature = (
    (
        tf.TensorSpec(shape=(None, img_size[0], img_size[1], 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None, img_size[0], img_size[1], 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None, img_size[0], img_size[1], 3), dtype=tf.float32),
    ),
    tf.TensorSpec(shape=(None, num_classes), dtype=tf.float32)
)

train_ds = tf.data.Dataset.from_generator(gen_train, output_signature=output_signature).prefetch(tf.data.AUTOTUNE)
val_ds   = tf.data.Dataset.from_generator(gen_val,   output_signature=output_signature).prefetch(tf.data.AUTOTUNE)

steps_per_epoch = train_flow.n // batch_size
val_steps = val_flow.n // batch_size

# ------------------ CHECKPOINTS ------------------
checkpoint_best = ModelCheckpoint(
    phase3_best,
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

checkpoint_all = ModelCheckpoint(
    phase3_epoch_save,
    verbose=1,
    save_freq="epoch"
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=4,
    min_lr=1e-8,
    verbose=1
)

early = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

csvlog = CSVLogger(log_path)

# ------------------ TRAIN PHASE-3 ------------------
print("\n🚀 Starting Phase-3 Training...\n")

model.fit(
    train_ds,
    epochs=phase3_epochs,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_ds,
    validation_steps=val_steps,
    callbacks=[checkpoint_best, checkpoint_all, reduce_lr, early, csvlog]
)

print("\n🎉 Phase-3 training complete!")


In [ ]:
# ---------------------------------------
# FULL PHASE-3 EVALUATION (ALL METRICS)
# ---------------------------------------

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import pandas as pd
import cv2
import skimage.color
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# ------------ PATHS ----------------
test_path = "/test"
model_path = "/mcrestnet_phase3.h5"
img_size = (224,224)

# ------------ LOAD MODEL ------------
model = load_model(model_path, compile=False)
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
print("✓ Phase-3 best model loaded!")

# ------------ PREPROCESSING ----------
def scale0to255(image):
    a = image.astype(np.float32)
    for i in range(3):
        mn, mx = np.min(a[:,:,i]), np.max(a[:,:,i])
        a[:,:,i] = ((a[:,:,i]-mn)/(mx-mn))*255 if mx!=mn else 0
    return a.astype(np.uint8)

def log_filter(image):
    g = cv2.GaussianBlur(image, (3,3), 0)
    lap = cv2.Laplacian(np.uint8(g), cv2.CV_64F)
    return np.uint8(np.clip(image + lap, 0, 255))

def to_lch(batch):
    out=[]
    for img in batch:
        lab = skimage.color.rgb2lab(log_filter(img))
        out.append(scale0to255(skimage.color.lab2lch(lab)))
    return np.stack(out)

def to_hsv(batch):
    out=[]
    for img in batch:
        hsv = skimage.color.rgb2hsv(log_filter(img))
        out.append(scale0to255(np.nan_to_num(hsv)))
    return np.stack(out)

# ---------- TEST GENERATOR ----------
test_aug = ImageDataGenerator()

test_flow = test_aug.flow_from_directory(
    test_path,
    target_size=img_size,
    batch_size=32,   # FAST evaluation
    class_mode='categorical',
    shuffle=False
)

# -------- FAST BATCHED EVALUATION --------
y_true = []
prob_list = []

steps = int(np.ceil(test_flow.n / 32))
test_flow.reset()

for _ in tqdm(range(steps)):
    x_batch, y_batch = next(test_flow)

    x_uint8 = x_batch.astype(np.uint8)

    lch_batch = to_lch(x_uint8)
    hsv_batch = to_hsv(x_uint8)

    x_rgb = tf.keras.applications.resnet.preprocess_input(x_batch.astype(np.float32))
    x_lch = tf.keras.applications.resnet.preprocess_input(lch_batch.astype(np.float32))
    x_hsv = tf.keras.applications.resnet.preprocess_input(hsv_batch.astype(np.float32))

    preds = model.predict([x_rgb, x_lch, x_hsv], verbose=0)

    prob_list.extend(preds)
    y_true.extend(np.argmax(y_batch, axis=1))

probs = np.array(prob_list)
y_pred = np.argmax(probs, axis=1)

print("✓ Phase-3 predictions complete!")

# -------- LABEL MAPPING --------
labels_map = {v:k for k,v in test_flow.class_indices.items()}
class_names = list(labels_map.values())

# -------- CLASSIFICATION REPORT --------
print("\n===== CLASSIFICATION REPORT (Phase-3) =====\n")
print(classification_report(y_true, y_pred, target_names=class_names))

# -------- CONFUSION MATRIX --------
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, cmap='Blues', fmt='d',
            xticklabels=class_names,
            yticklabels=class_names)
plt.title("Confusion Matrix (Phase-3)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# -------- ROC + AUC --------
y_true_onehot = tf.keras.utils.to_categorical(y_true, num_classes=len(class_names))

plt.figure(figsize=(8,6))
for i, cname in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_true_onehot[:,i], probs[:,i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{cname} (AUC = {roc_auc:.3f})")

plt.plot([0,1], [0,1], linestyle='--')
plt.title("ROC Curves (Phase-3)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid()
plt.show()

# -------- SAVE CSV --------
df = pd.DataFrame(probs, columns=[f"Conf_{c}" for c in class_names])
df["True"] = [labels_map[i] for i in y_true]
df["Pred"] = [labels_map[i] for i in y_pred]

df.to_csv("phase3_full_metrics.csv", index=False)
print("\n✓ Saved → phase3_full_metrics.csv")

print("\n🎉 Phase-3 evaluation complete!")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_path = "/phase3_training_log.csv"

try:
    history = pd.read_csv(log_path)
    print("✓ Training log loaded!")

    # ---- LOSS CURVE ----
    plt.figure(figsize=(7,5))
    plt.plot(history["loss"], label="Train Loss")
    plt.plot(history["val_loss"], label="Val Loss")
    plt.title("Phase-3 Loss Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid()
    plt.show()

    # ---- ACCURACY CURVE ----
    plt.figure(figsize=(7,5))
    plt.plot(history["accuracy"], label="Train Acc")
    plt.plot(history["val_accuracy"], label="Val Acc")
    plt.title("Phase-3 Accuracy Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid()
    plt.show()

except:
    print("⚠ phase3_training_log.csv not found. Train Phase-3 again or check path.")


In [ ]:
from tensorflow.keras.models import load_model

model_path = "/mcrestnet_phase3.h5"
model = load_model(model_path, compile=False)
import numpy as np
import cv2
import skimage.color

def scale0to255(image):
    a = image.astype(np.float32)
    for i in range(3):
        mn, mx = np.min(a[:,:,i]), np.max(a[:,:,i])
        a[:,:,i] = ((a[:,:,i]-mn)/(mx-mn))*255 if mx!=mn else 0
    return a.astype(np.uint8)

def log_filter(img):
    g = cv2.GaussianBlur(img, (3,3), 0)
    lap = cv2.Laplacian(np.uint8(g), cv2.CV_64F)
    return np.uint8(np.clip(img + lap, 0, 255))

def to_lch(batch):
    out=[]
    for img in batch:
        lab = skimage.color.rgb2lab(log_filter(img))
        out.append(scale0to255(skimage.color.lab2lch(lab)))
    return np.stack(out)

def to_hsv(batch):
    out=[]
    for img in batch:
        hsv = skimage.color.rgb2hsv(log_filter(img))
        out.append(scale0to255(np.nan_to_num(hsv)))
    return np.stack(out)



import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt

def grad_cam_resnet50_phase3(model,
                             img_rgb_uint8,
                             layer_name,


    # --- 1. Preprocess all 3 branches ---
    x = img_rgb_uint8.astype(np.float32)
    x = np.expand_dims(x, axis=0)

    # Convert using your LCH/HSV preprocessing
    lch = to_lch(img_rgb_uint8[np.newaxis, ...])
    hsv = to_hsv(img_rgb_uint8[np.newaxis, ...])

    x_rgb = tf.keras.applications.resnet.preprocess_input(x)
    x_lch = tf.keras.applications.resnet.preprocess_input(lch.astype(np.float32))
    x_hsv = tf.keras.applications.resnet.preprocess_input(hsv.astype(np.float32))

    # --- 2. Get the correct layer ---
    target_layer = model.get_layer(layer_name)

    # --- 3. Build grad model ---
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[target_layer.output, model.output]
    )

    # --- 4. Compute gradient ---
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model([x_rgb, x_lch, x_hsv])
        pred_index = tf.argmax(preds[0])
        class_output = preds[:, pred_index]

    grads = tape.gradient(class_output, conv_out)

    # --- 5. Compute weights ---
    weights = tf.reduce_mean(grads, axis=(0,1,2))
    cam = tf.reduce_sum(tf.multiply(weights, conv_out[0]), axis=-1).numpy()

    # --- 6. Normalize CAM ---
    cam = np.maximum(cam, 0)
    cam = cam / (cam.max() + 1e-8)

    # Resize to image size
    cam = cv2.resize(cam, (W, H))

    return cam


In [ ]:
img_path = "/prev.jpg"

orig = cv2.imread(img_path)
orig = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)
orig_resized = cv2.resize(orig, (224,224))

cam = grad_cam_resnet50_phase3(
    model,
    orig_resized,
    layer_name="r1_layer_174"
)


plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(orig_resized)
plt.title("Original")

plt.subplot(1,2,2)
plt.imshow(orig_resized)
plt.imshow(cam, cmap="jet", alpha=0.6)
plt.title("Grad-CAM")
plt.show()
